# Actividad 15. Implementación de técnicas de Programación Dinámica

**Materia:** Lenguajes de Computación · Otoño 2026

| Nombre completo | No. de cuenta |
|---|---|
| Angel Rugerio Jiménez | 201720 |

**Objetivo:** implementar las técnicas de **Memoización (Top-Down)** y **Tabulación (Bottom-Up)** para el problema del **Corte de una Varilla**, y comparar su rendimiento contra la solución **recursiva pura** con distintos tamaños de varilla.

**Estructura:** problema y solución recursiva (de la clase) → memoización (de la clase) → **tabulación** → **reconstrucción de los cortes** → verificación → **benchmark** → análisis.

## El Problema del corte de varillas (Rod Cutting Problem)

El problema consiste en que considerando una varilla de *n* pulgadas y una tabla de precios $p_i$ por longitud, donde *i* = {1, 2, ..., n}, se pueda determinar el máximo ingreso por los cortes realizados o no, en la varilla.

Considerando el precio de cada corte en la varilla, es posible que no necesariamente se considere un corte.

### Tabla de precios por tamaño de corte

| Longitud (i) | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Precio | \$1 | \$5 | \$8 | \$9 | \$10 | \$17 | \$17 | \$20 | \$24 | \$30 |

Una varilla de *n* pulgadas se puede cortar de $2^{n-1}$ formas distintas, ya que en cada una de las $n-1$ posiciones *i* = {1, 2, ..., n-1} se decide si se corta o no.

### Solución recursiva (pura)

In [1]:
# importación de librerías
import time
import sys

In [2]:
# Ampliación del límite de profundidad de recursión para los algoritmos recursivos
sys.setrecursionlimit(2000)

In [3]:
# Definición de la entrada constante
PRECIOS = (0, 1, 5, 8, 9, 10, 17, 17, 20, 24, 30)

In [4]:
# Implementación de la función recursiva pura
def cut_rod(p: list[int], n:int) -> int:
    """
    Función recursiva para calcular el máximo ingreso por los cortes realizados en una varilla de longitud n.
    """
    if n == 0:
        return 0
    q = -1
    for i in range(1, n+1):
      # Evaluar todas las opciones de corte inicial de tamaño i e invocando de manera recursiva a la función para calcular el máximo ingreso con el remanente
      precio_i = p[i] if i < len(p) else 0
      q = max(q, precio_i + cut_rod(p, n-i))
    return q

### Implementación de las técnicas de programación dinámica

Como se pudo indentificar previamente, en el peor de los casos el problema del cálculo de la máxima ganancia en el corte de una varilla, está dado por $O(2^{n})$.

Justamente la programación dinámica, busca evitar la redundancia en el cálculo de algunos cortes que previamente ya fueron evaluados a través de la implementación ya sea de una memoria caché (arreglo) o bien de una tabla.

#### Memoización (Top-Down)

Esta técnica reutiliza la misma función recursiva pura y agrega una caché para verificar si el resultado de un cálculo ya fue evaluado previamente, en cuyo caso recupera el valor almacenado ($O(1)$).

In [5]:
# Definición de la función de memoización
def cut_rod_memoized(p: list[int], n:int) -> int:
    """
    Función recursiva para calcular el máximo ingreso por los cortes realizados en una varilla de longitud n, implementando una función auxiliar de memoización
    """
    # Creación de un arreglo "r" de tamaño n+1 e inicilizado a -1 (valores desconocidos/no calculados)
    r = [-1] * (n+1)
    return cut_rod_aux(p, n, r)

def cut_rod_aux(p: list[int], n:int, r: list[int]) -> int:
    """
    Función auxiliar para la implementación de la técnica de memoización para verificar si el resultado de un cálculo ya fue evaluado previamente, en cuyo caso recupera el valor almacenado
    """
    # Verificación de resultado en memoria
    if r[n] >= 0:
      return r[n]

    # Caso de finalización/retorno
    if n==0:
      q=0
    else:
      q = -1
      for i in range(1, n+1):
        # Evaluar todas las opciones de corte inicial de tamaño i e invocando de manera recursiva a la función para calcular el máximo ingreso con el remanente
        precio_i = p[i] if i < len(p) else 0
        q = max(q, precio_i + cut_rod_aux(p, n-i, r))

    # Almacenamiento del resultado en memoria
    r[n] = q
    return q

#### Implementación de la técnica tabulación (Bottom-Up)

Esta técnica elimina la recursión: resuelve los subproblemas **de menor a mayor**, guardándolos en una tabla `r`, de modo que cuando se calcula `r[j]` todos los `r[j-i]` que necesita ya están calculados.

$$r_0 = 0, \qquad r_j = \max_{1 \le i \le j} \left( p_i + r_{j-i} \right), \quad j = 1, 2, \dots, n$$

- **Tiempo:** dos ciclos anidados, $\sum_{j=1}^{n} j = \frac{n(n+1)}{2}$ iteraciones $\Rightarrow O(n^2)$. Es la misma complejidad que la memoización, pero sin el costo de las llamadas recursivas.
- **Espacio:** la tabla `r` de $n+1$ casillas $\Rightarrow O(n)$, sin pila de recursión.

In [6]:
# Implementación de la función con tabulación
def cut_rod_bottom_up(p: list[int], n: int) -> int:
    """
    Función iterativa para calcular el máximo ingreso por los cortes realizados en una varilla de longitud n, llenando una tabla de abajo hacia arriba
    """
    # Tabla "r" de tamaño n+1; r[0] = 0 porque una varilla de longitud 0 no genera ingreso
    r = [0] * (n+1)
    for j in range(1, n+1):
        # Calcular el máximo ingreso de una varilla de longitud j con los subproblemas ya resueltos
        q = -1
        for i in range(1, j+1):
            precio_i = p[i] if i < len(p) else 0
            q = max(q, precio_i + r[j-i])
        r[j] = q
    return r[n]

#### Reconstrucción de los cortes

Las tres funciones anteriores solo regresan el **ingreso máximo**. Para saber **qué cortes** lo producen, cada técnica se adecua para guardar, además del ingreso, la **longitud del primer corte** de la solución óptima:

- **Recursivo:** cada llamada regresa la tupla `(ingreso, cortes)`, con la lista de cortes del remanente más el corte actual.
- **Memoización y tabulación:** un arreglo `s` donde `s[j]` es el tamaño del primer corte óptimo para una varilla de longitud `j`. Los cortes se reconstruyen recorriendo `s`: se corta `s[n]`, queda `n - s[n]`, se corta `s[n - s[n]]`, y así hasta llegar a 0.

> Nota: `PRECIOS` solo define longitudes del 1 al 10; una pieza de más de 10 pulgadas tiene precio 0, así que las varillas largas siempre terminan cortadas en piezas de 10 o menos.

In [7]:
# Versión recursiva pura que también regresa los cortes
def cut_rod_cortes(p: list[int], n: int) -> tuple[int, list[int]]:
    """
    Regresa (ingreso máximo, lista de cortes) para una varilla de longitud n, de forma recursiva pura
    """
    if n == 0:
        return 0, []
    q, mejores_cortes = -1, []
    for i in range(1, n+1):
        precio_i = p[i] if i < len(p) else 0
        ingreso_resto, cortes_resto = cut_rod_cortes(p, n-i)
        if precio_i + ingreso_resto > q:
            q = precio_i + ingreso_resto
            mejores_cortes = [i] + cortes_resto
    return q, mejores_cortes


def reconstruir_cortes(s: list[int], n: int) -> list[int]:
    """
    Recorre el arreglo s (primer corte óptimo de cada longitud) para obtener la lista de cortes de una varilla de longitud n
    """
    cortes = []
    while n > 0:
        cortes.append(s[n])
        n = n - s[n]
    return cortes

In [8]:
# Versión con memoización que también regresa los cortes
def cut_rod_memoized_cortes(p: list[int], n: int) -> tuple[int, list[int]]:
    """
    Regresa (ingreso máximo, lista de cortes) usando memoización; s[j] guarda el primer corte óptimo de la longitud j
    """
    r = [-1] * (n+1)
    s = [0] * (n+1)
    q = cut_rod_aux_cortes(p, n, r, s)
    return q, reconstruir_cortes(s, n)

def cut_rod_aux_cortes(p: list[int], n: int, r: list[int], s: list[int]) -> int:
    # Verificación de resultado en memoria
    if r[n] >= 0:
        return r[n]
    if n == 0:
        q = 0
    else:
        q = -1
        for i in range(1, n+1):
            precio_i = p[i] if i < len(p) else 0
            candidato = precio_i + cut_rod_aux_cortes(p, n-i, r, s)
            if candidato > q:
                q = candidato
                s[n] = i  # mejor primer corte encontrado hasta ahora
    r[n] = q
    return q


# Versión con tabulación que también regresa los cortes
def cut_rod_bottom_up_cortes(p: list[int], n: int) -> tuple[int, list[int]]:
    """
    Regresa (ingreso máximo, lista de cortes) usando tabulación; s[j] guarda el primer corte óptimo de la longitud j
    """
    r = [0] * (n+1)
    s = [0] * (n+1)
    for j in range(1, n+1):
        q = -1
        for i in range(1, j+1):
            precio_i = p[i] if i < len(p) else 0
            if precio_i + r[j-i] > q:
                q = precio_i + r[j-i]
                s[j] = i
        r[j] = q
    return r[n], reconstruir_cortes(s, n)

#### Verificación

Antes de medir, se comprueba que las seis funciones coinciden para todas las longitudes de 0 a 20, y que los cortes reportados son consistentes: **suman `n` pulgadas** y **sus precios suman el ingreso máximo**. (Se usa hasta 20 porque la recursiva pura con cortes ya tarda varios segundos a partir de ahí.)

In [9]:
def precio(p: list[int], i: int) -> int:
    return p[i] if i < len(p) else 0

for n in range(0, 21):
    ingresos = {
        cut_rod(PRECIOS, n),
        cut_rod_memoized(PRECIOS, n),
        cut_rod_bottom_up(PRECIOS, n),
    }
    for funcion in (cut_rod_cortes, cut_rod_memoized_cortes, cut_rod_bottom_up_cortes):
        ingreso, cortes = funcion(PRECIOS, n)
        assert sum(cortes) == n, (funcion.__name__, n, cortes)
        assert sum(precio(PRECIOS, c) for c in cortes) == ingreso, (funcion.__name__, n, cortes)
        ingresos.add(ingreso)
    assert len(ingresos) == 1, (n, ingresos)

print("Las 6 funciones coinciden para n = 0..20 y sus cortes son consistentes.")
for n in (4, 7, 10):
    print(f"n = {n:2d}: ingreso = {cut_rod_bottom_up(PRECIOS, n)}, cortes = {cut_rod_bottom_up_cortes(PRECIOS, n)[1]}")

Las 6 funciones coinciden para n = 0..20 y sus cortes son consistentes.
n =  4: ingreso = 10, cortes = [2, 2]
n =  7: ingreso = 18, cortes = [1, 6]
n = 10: ingreso = 30, cortes = [10]


#### Benchmark de las 3 técnicas

Elabora una tabla que compare los tiempos para el cálculo de los ingresos máximos utilizando las tres técnicas, y adecuando las funciones para conocer cuáles serían los cortes.

**Metodología de medición:**

- Se mide cada técnica con `time.perf_counter()` (reloj de alta resolución) y el módulo `timeit` de la biblioteca estándar.
- Las funciones medidas son las tres que solo calculan el ingreso (`cut_rod`, `cut_rod_memoized`, `cut_rod_bottom_up`); los **cortes** se obtienen después con la versión de tabulación adecuada, para que construir listas no altere los tiempos.
- Una ejecución de memoización o tabulación tarda microsegundos, menos que el ruido del sistema. Por eso, cuando una corrida dura menos de 1 s, se repite en bloques (`timeit.autorange`) y se reporta el **mínimo de 5 bloques** dividido entre el número de ejecuciones del bloque. Cuando una corrida dura 1 s o más (la recursiva pura con $n \ge 25$) se reporta esa única corrida.
- En cada renglón se comprueba que las tres técnicas regresan el mismo ingreso.
- La recursiva pura con $n = 30$ tarda más de 2 minutos: la celda del benchmark completa tarda alrededor de 3 minutos.

In [10]:
import os
import platform

def leer_cpu() -> str:
    # Modelo del procesador desde /proc/cpuinfo (solo Linux)
    try:
        with open("/proc/cpuinfo", encoding="utf-8") as f:
            for linea in f:
                if linea.startswith("model name"):
                    return linea.split(":", 1)[1].strip()
    except OSError:
        pass
    return platform.processor() or "no disponible"

print(f"Procesador : {leer_cpu()} ({os.cpu_count()} hilos)")
print(f"Sistema    : {platform.system()} {platform.release()}")
print(f"Python     : {platform.python_version()} ({platform.python_implementation()})")

Procesador : Intel(R) Core(TM) Ultra 5 228V (8 hilos)
Sistema    : Linux 7.0.0-42-generic
Python     : 3.11.15 (CPython)


In [11]:
import timeit

LONGITUDES = [5, 10, 15, 20, 25, 28, 30]

def medir(funcion, n: int, repeticiones: int = 5) -> tuple[float, int]:
    """
    Regresa (segundos por ejecución, ingreso) de funcion(PRECIOS, n)
    """
    inicio = time.perf_counter()
    ingreso = funcion(PRECIOS, n)
    primera = time.perf_counter() - inicio
    if primera >= 1.0:
        return primera, ingreso  # corrida lenta: basta una medición
    timer = timeit.Timer(lambda: funcion(PRECIOS, n))
    numero, _ = timer.autorange()
    return min(timer.repeat(repeticiones, numero)) / numero, ingreso

resultados = []
for n in LONGITUDES:
    t_rec, ingreso_rec = medir(cut_rod, n)
    t_memo, ingreso_memo = medir(cut_rod_memoized, n)
    t_tab, ingreso_tab = medir(cut_rod_bottom_up, n)
    assert ingreso_rec == ingreso_memo == ingreso_tab, (n, ingreso_rec, ingreso_memo, ingreso_tab)
    _, cortes = cut_rod_bottom_up_cortes(PRECIOS, n)
    resultados.append({"n": n, "recursivo": t_rec, "memoizacion": t_memo,
                       "tabulacion": t_tab, "ingreso": ingreso_tab, "cortes": cortes})
    print(f"n = {n:2d} listo (recursivo: {t_rec:.6f} s)")

n =  5 listo (recursivo: 0.000004 s)


n = 10 listo (recursivo: 0.000117 s)


n = 15 listo (recursivo: 0.003721 s)


n = 20 listo (recursivo: 0.119428 s)


n = 25 listo (recursivo: 3.921134 s)


n = 28 listo (recursivo: 34.503307 s)


n = 30 listo (recursivo: 128.499122 s)


In [12]:
from IPython.display import Markdown, display

def formato_tiempo(segundos: float) -> str:
    if segundos >= 1:
        return f"{segundos:.2f} s"
    if segundos >= 1e-3:
        return f"{segundos * 1e3:.2f} ms"
    return f"{segundos * 1e6:.2f} µs"

filas = ["| Longitud de la varilla | Recursivo | Memoización | Tabulación | Ingreso Máximo | Cortes |",
         "| --- | --- | --- | --- | --- | --- |"]
for r in resultados:
    cortes = " + ".join(str(c) for c in r["cortes"])
    filas.append(f"| {r['n']} | {formato_tiempo(r['recursivo'])} | {formato_tiempo(r['memoizacion'])} | "
                 f"{formato_tiempo(r['tabulacion'])} | \\${r['ingreso']} | {cortes} |")
tabla = "\n".join(filas)
display(Markdown(tabla))

| Longitud de la varilla | Recursivo | Memoización | Tabulación | Ingreso Máximo | Cortes |
| --- | --- | --- | --- | --- | --- |
| 5 | 3.55 µs | 1.94 µs | 1.58 µs | \$13 | 2 + 3 |
| 10 | 116.80 µs | 6.46 µs | 5.08 µs | \$30 | 10 |
| 15 | 3.72 ms | 12.98 µs | 10.10 µs | \$43 | 2 + 3 + 10 |
| 20 | 119.43 ms | 23.68 µs | 17.58 µs | \$60 | 10 + 10 |
| 25 | 3.92 s | 33.83 µs | 25.53 µs | \$73 | 2 + 3 + 10 + 10 |
| 28 | 34.50 s | 43.43 µs | 32.97 µs | \$82 | 2 + 6 + 10 + 10 |
| 30 | 128.50 s | 46.93 µs | 36.31 µs | \$90 | 10 + 10 + 10 |

In [13]:
# Cuántas veces más lenta es cada técnica respecto a la tabulación, y crecimiento de la recursiva
print(f"{'n':>3} | {'rec / tab':>12} | {'memo / tab':>10} | {'rec(n) / rec(n anterior)':>24}")
anterior = None
for r in resultados:
    crecimiento = f"{r['recursivo'] / anterior['recursivo']:.1f}x (n +{r['n'] - anterior['n']})" if anterior else "-"
    print(f"{r['n']:>3} | {r['recursivo'] / r['tabulacion']:>11,.0f}x | {r['memoizacion'] / r['tabulacion']:>9.2f}x | {crecimiento:>24}")
    anterior = r

  n |    rec / tab | memo / tab | rec(n) / rec(n anterior)
  5 |           2x |      1.23x |                        -
 10 |          23x |      1.27x |             32.9x (n +5)
 15 |         368x |      1.29x |             31.9x (n +5)
 20 |       6,792x |      1.35x |             32.1x (n +5)
 25 |     153,567x |      1.32x |             32.8x (n +5)
 28 |   1,046,463x |      1.32x |              8.8x (n +3)
 30 |   3,538,619x |      1.29x |              3.7x (n +2)


#### Resultados

Medido el 2026-11-04 en un Intel Core Ultra 5 228V (8 hilos), Linux, CPython 3.11. Los tiempos absolutos cambian con el equipo y la versión de Python; lo que se compara es **cómo crecen** con $n$.

| Longitud de la varilla | Recursivo | Memoización | Tabulación | Ingreso Máximo | Cortes |
| --- | --- | --- | --- | --- | --- |
| 5 | 3.55 µs | 1.94 µs | 1.58 µs | \$13 | 2 + 3 |
| 10 | 116.80 µs | 6.46 µs | 5.08 µs | \$30 | 10 |
| 15 | 3.72 ms | 12.98 µs | 10.10 µs | \$43 | 2 + 3 + 10 |
| 20 | 119.43 ms | 23.68 µs | 17.58 µs | \$60 | 10 + 10 |
| 25 | 3.92 s | 33.83 µs | 25.53 µs | \$73 | 2 + 3 + 10 + 10 |
| 28 | 34.50 s | 43.43 µs | 32.97 µs | \$82 | 2 + 6 + 10 + 10 |
| 30 | 128.50 s | 46.93 µs | 36.31 µs | \$90 | 10 + 10 + 10 |

### Análisis

1. **Recursiva pura: crecimiento exponencial $O(2^n)$.** Cada 5 pulgadas más, el tiempo se multiplica por $\approx 32 = 2^5$; de 25 a 28 por $\approx 8.8$ ($2^3 = 8$) y de 28 a 30 por $\approx 3.7$ ($2^2 = 4$). Es justo lo que predice el análisis: `cut_rod(n)` hace $2^n$ llamadas porque resuelve una y otra vez los mismos subproblemas. Con $n = 30$ ya pasa de 2 minutos; con $n = 40$ serían del orden de $2^{10} \approx 1000$ veces más, casi 1.5 días.
2. **Memoización y tabulación: crecimiento polinomial $O(n^2)$.** Ambas resuelven cada subproblema **una sola vez**, así que de $n = 5$ a $n = 30$ solo pasan de unos 2 µs a unos 40 µs. Para $n = 30$ son **más de 3 millones de veces** más rápidas que la recursiva pura.
3. **Tabulación vs. memoización.** Tienen la misma complejidad, pero la tabulación es consistentemente **~1.3 veces más rápida**: la memoización paga el costo de cada llamada recursiva (crear el marco en la pila, pasar argumentos, revisar la caché), mientras que la tabulación son dos ciclos `for` sobre un arreglo. Además, la tabulación no depende del límite de recursión de Python (`sys.setrecursionlimit`): la memoización con $n$ de algunos miles fallaría con `RecursionError` y la tabulación no.
4. **Cortes óptimos.** La pieza de 10 pulgadas es la de mejor precio por pulgada (\$3.00), seguida de la de 6 (\$2.83) y la de 2 (\$2.50). Por eso las soluciones usan tantas piezas de 10 como pueden y completan el resto con piezas pequeñas: con $n = 28$, $10 + 10 + 6 + 2$ pulgadas dan \$82. Como las piezas de más de 10 pulgadas no tienen precio, toda varilla más larga se corta.
5. **Cuándo conviene cada técnica.** La memoización solo calcula los subproblemas que realmente se necesitan, lo que ayuda cuando no todos se usan; en este problema se usan todos ($r_0, \dots, r_n$), así que la tabulación es la mejor opción: más rápida, sin recursión y con el mismo resultado.